# Data Preprocessing

### Install Library

In [1]:
pip install -q imbalanced-learn

### Import Library

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

### Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1. Data Cleaning

In [4]:
# Definisi path input & output
DATA_PATH = '/content/drive/MyDrive/Fraud Detection/data/creditcard.csv'
OUTPUT_DIR = '/content/drive/MyDrive/Fraud Detection/data/processed'

# Buat folder output jika belum ada
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Konstanta
RANDOM_STATE = 42
LABEL = 'Class'
v_cols = [f'V{i}' for i in range(1, 29)]

# Load dataset
df = pd.read_csv(DATA_PATH)
print(f'Dataset dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom')
print()
print(df.head())

Dataset dimuat: 284,807 baris, 31 kolom

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.20

In [5]:
# Cek baris duplikat
duplicate_count = df.duplicated().sum()
print(f'Jumlah baris duplikat: {duplicate_count:,} ({duplicate_count / len(df) * 100:.2f}% dari data)')

# Hapus duplikat agar tiap transaksi unik
df_clean = df.drop_duplicates().reset_index(drop=True)
print()
print(f'Bentuk sebelum pembersihan : {df.shape[0]:,} baris')
print(f'Bentuk sesudah pembersihan : {df_clean.shape[0]:,} baris')

Jumlah baris duplikat: 1,081 (0.38% dari data)

Bentuk sebelum pembersihan : 284,807 baris
Bentuk sesudah pembersihan : 283,726 baris


In [7]:
# Cek missing values
missing = df_clean.isnull().sum()
total_missing = missing.sum()
print(f'Total missing values: {total_missing}')

if total_missing > 0:
    print('\nKolom dengan missing values:')
    print(missing[missing > 0])
    # Imputasi sederhana dengan median jika diperlukan
    df_clean = df_clean.fillna(df_clean.median(numeric_only=True))
    print('\nImputasi median selesai dilakukan.')
else:
    print('Tidak ada missing values.')

Total missing values: 0
Tidak ada missing values.


In [8]:
# Validasi integritas data
print('Validasi integritas data:')
print(f'1. Semua kolom numerik   : {all(pd.api.types.is_numeric_dtype(dt) for dt in df_clean.dtypes)}')
print(f'2. Class hanya berisi 0/1: {set(df_clean[LABEL].unique()).issubset({0, 1})}')
print(f'3. Amount >= 0           : {(df_clean["Amount"] >= 0).all()}')
print(f'4. Time   >= 0           : {(df_clean["Time"] >= 0).all()}')
print(f'5. Tidak ada NaN         : {not df_clean.isnull().any().any()}')

assert not df_clean.isnull().any().any(), 'Masih ada NaN pada data.'
print()
print('Data bersih dan siap diproses.')

Validasi integritas data:
1. Semua kolom numerik   : True
2. Class hanya berisi 0/1: True
3. Amount >= 0           : True
4. Time   >= 0           : True
5. Tidak ada NaN         : True

Data bersih dan siap diproses.


### 2. Three-Way Stratified Split (70:15:15)

In [9]:
# Pisahkan fitur (X) dan label (y)
X = df_clean.drop(columns=[LABEL])
y = df_clean[LABEL]

# STEP 1: Split 70-30 (train vs temp)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)

print(f'Train: {len(X_train):,} ({len(X_train) / len(df_clean) * 100:.1f}%)')
print(f'Temp : {len(X_temp):,} ({len(X_temp) / len(df_clean) * 100:.1f}%)')

Train: 198,608 (70.0%)
Temp : 85,118 (30.0%)


In [10]:
# STEP 2: Split temp 50-50 (val vs test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)

print('Komposisi final split (70:15:15):')
print(f'Train: {len(X_train):,} ({len(X_train) / len(df_clean) * 100:.1f}%)')
print(f'Val  : {len(X_val):,} ({len(X_val) / len(df_clean) * 100:.1f}%)')
print(f'Test : {len(X_test):,} ({len(X_test) / len(df_clean) * 100:.1f}%)')

Komposisi final split (70:15:15):
Train: 198,608 (70.0%)
Val  : 42,559 (15.0%)
Test : 42,559 (15.0%)


In [11]:
# Validasi stratifikasi: proporsi fraud harus konsisten di semua split
def fraud_rate(ser):
    return (ser == 1).mean() * 100

print('Fraud rate per split (target ~ sama dengan data asli):')
print(f'Original : {fraud_rate(y):.4f}%')
print(f'Train    : {fraud_rate(y_train):.4f}%')
print(f'Val      : {fraud_rate(y_val):.4f}%')
print(f'Test     : {fraud_rate(y_test):.4f}%')

Fraud rate per split (target ~ sama dengan data asli):
Original : 0.1667%
Train    : 0.1667%
Val      : 0.1668%
Test     : 0.1668%


### 3. Feature Normalization (Prevent Data Leakage)

In [12]:
# Fitur engineering dibuat SEBELUM scaling (log transform harus mendahului standardisasi).
# Catatan: seluruh operasi di bawah bersifat per-baris (tidak memakai statistik global),
# sehingga aman dari data leakage meskipun diterapkan sebelum splitting.
def add_engineered_features(df_split):
    df_split = df_split.copy()
    df_split['Amount_log'] = np.log1p(df_split['Amount'])
    df_split['Time_hour'] = (df_split['Time'] // 3600) % 24
    df_split['Time_day'] = df_split['Time'] // 86400
    return df_split

X_train_fe = add_engineered_features(X_train)
X_val_fe = add_engineered_features(X_val)
X_test_fe = add_engineered_features(X_test)

# Fitur akhir model: V1-V28 (PCA, sudah ternormalisasi) + fitur engineering
FEATURES = v_cols + ['Amount_log', 'Time_hour', 'Time_day']
print(f'Jumlah fitur model: {len(FEATURES)}')
print(FEATURES)

Jumlah fitur model: 31
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount_log', 'Time_hour', 'Time_day']


In [13]:
# SCALER HANYA DI-FIT PADA DATA TRAIN (mencegah data leakage)
scaler = StandardScaler()
scaler.fit(X_train_fe[FEATURES])

scaler_info = pd.DataFrame({'mean': scaler.mean_, 'std': scaler.scale_}, index=FEATURES)
print('StandardScaler di-fit hanya pada TRAIN.')
print('Parameter scaler (contoh 5 fitur pertama):')
print(scaler_info.head().round(3))

StandardScaler di-fit hanya pada TRAIN.
Parameter scaler (contoh 5 fitur pertama):
     mean    std
V1  0.005  1.947
V2 -0.004  1.646
V3  0.002  1.506
V4  0.000  1.419
V5  0.005  1.357


In [14]:
# Transform train, val, test menggunakan scaler yang sama (dari train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train_fe[FEATURES]), columns=FEATURES)
X_val_scaled = pd.DataFrame(scaler.transform(X_val_fe[FEATURES]), columns=FEATURES)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_fe[FEATURES]), columns=FEATURES)

train_stats = pd.DataFrame({'mean': X_train_scaled.mean(), 'std': X_train_scaled.std()})
print('Distribusi train setelah scaling (mean ~0, std ~1):')
print(train_stats.round(4).describe().loc[['mean', 'std']])

print()
print('Contoh X_train_scaled (5 baris pertama):')
print(X_train_scaled.head())

Distribusi train setelah scaling (mean ~0, std ~1):
      mean  std
mean   0.0  1.0
std    0.0  0.0

Contoh X_train_scaled (5 baris pertama):
         V1        V2        V3        V4        V5        V6        V7  \
0  1.095958 -1.443147 -0.391914 -1.491608 -0.115548  3.722286 -2.615588   
1 -1.107213  0.274431  0.386511  0.832013  0.129495 -0.906210 -0.090235   
2 -3.084038 -3.562390 -0.805080  1.574006 -0.685674  0.015266  0.008751   
3  1.035377 -0.322817 -0.277757 -0.087702 -0.135457  0.693905 -0.871156   
4  0.074166  0.493288 -0.393781 -0.359307  0.840218 -0.842423  0.779326   

         V8        V9       V10  ...       V22       V23       V24       V25  \
0  1.199538  0.397404  1.110886  ...  0.868962  0.389884  1.220372 -0.603450   
1  0.657230 -1.313138 -0.448124  ... -1.199743 -0.066157  0.828187 -0.379224   
2  0.775201 -0.823883  0.219241  ... -1.042448  0.742917 -1.236684 -0.466150   
3  0.271032  0.982595  0.112422  ...  0.289211  0.356929 -0.517064 -0.880460   
4 -0.19

### 4. Feature Engineering (Optional)

In [15]:
# Validasi fitur engineered yang sudah dibuat pada langkah 3
print('1) Amount_log mengatasi right skew:')
print(f'   Skew Amount     : {X_train["Amount"].skew():.2f}')
print(f'   Skew Amount_log : {X_train_fe["Amount_log"].skew():.2f}')
print()

print('2) Time_hour: distribusi jam transaksi')
print(X_train_fe['Time_hour'].value_counts().sort_index())
print()

print('3) Time_day: indeks hari (0 = hari pertama, 1 = hari kedua)')
print(X_train_fe['Time_day'].value_counts().sort_index())

1) Amount_log mengatasi right skew:
   Skew Amount     : 12.94
   Skew Amount_log : 0.16

2) Time_hour: distribusi jam transaksi
Time_hour
0.0      5316
1.0      2911
2.0      2322
3.0      2422
4.0      1539
5.0      2076
6.0      2861
7.0      5062
8.0      7170
9.0     11075
10.0    11640
11.0    11754
12.0    10761
13.0    10671
14.0    11551
15.0    11478
16.0    11448
17.0    11294
18.0    11830
19.0    11007
20.0    11754
21.0    12280
22.0    10812
23.0     7574
Name: count, dtype: int64

3) Time_day: indeks hari (0 = hari pertama, 1 = hari kedua)
Time_day
0.0    100773
1.0     97835
Name: count, dtype: int64


### 5. SMOTE - Only on TRAIN Set

In [16]:
# SMOTE HANYA diterapkan pada TRAIN SET agar model bisa belajar pola fraud secara seimbang
print('Distribusi kelas TRAIN sebelum SMOTE:')
print(y_train.value_counts())

smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Konversi label SMOTE ke Series untuk konsistensi
y_train_smote = pd.Series(y_train_smote, name=LABEL)

print()
print('Distribusi kelas TRAIN sesudah SMOTE (1:1 seimbang):')
print(y_train_smote.value_counts())

Distribusi kelas TRAIN sebelum SMOTE:
Class
0    198277
1       331
Name: count, dtype: int64

Distribusi kelas TRAIN sesudah SMOTE (1:1 seimbang):
Class
0    198277
1    198277
Name: count, dtype: int64


In [18]:
# VERIFIKASI: SMOTE TIDAK diterapkan pada val & test tetap sesuai kondisi dunia nyata
print('Fraud rate setelah data diprepare:')
print(f'  Train (setelah SMOTE) : {fraud_rate(y_train_smote):.1f}%')
print(f'  Val  (asli)           : {fraud_rate(y_val):.4f}%')
print(f'  Test (asli)           : {fraud_rate(y_test):.4f}%')

Fraud rate setelah data diprepare:
  Train (setelah SMOTE) : 50.0%
  Val  (asli)           : 0.1668%
  Test (asli)           : 0.1668%


### 6. Data Preparation for Models

In [19]:
# Ringkasan artefak yang akan dipakai di notebook 3-5
print('Artefak data siap untuk pemodelan:')
print()
print('Untuk perbandingan model (tanpa SMOTE):')
print(f'  X_train_scaled : {X_train_scaled.shape}')
print(f'  y_train        : {y_train.shape}')
print()
print('Untuk training model utama (dengan SMOTE):')
print(f'  X_train_smote  : {X_train_smote.shape}')
print(f'  y_train_smote  : {y_train_smote.shape}')
print()
print('Untuk validasi & testing (imbalanced, realistis):')
print(f'  X_val_scaled   : {X_val_scaled.shape}')
print(f'  X_test_scaled  : {X_test_scaled.shape}')
print(f'  y_val          : {y_val.shape}')
print(f'  y_test         : {y_test.shape}')
print()
print(f'Scaler: {type(scaler).__name__} (siap dipakai untuk transformasi data baru)')

Artefak data siap untuk pemodelan:

Untuk perbandingan model (tanpa SMOTE):
  X_train_scaled : (198608, 31)
  y_train        : (198608,)

Untuk training model utama (dengan SMOTE):
  X_train_smote  : (396554, 31)
  y_train_smote  : (396554,)

Untuk validasi & testing (imbalanced, realistis):
  X_val_scaled   : (42559, 31)
  X_test_scaled  : (42559, 31)
  y_val          : (42559,)
  y_test         : (42559,)

Scaler: StandardScaler (siap dipakai untuk transformasi data baru)


### 7. Validation & Save

In [20]:
# CEK DATA LEAKAGE
print('=== CEK DATA LEAKAGE ===')
print('1. StandardScaler di-fit hanya pada TRAIN         : OK')
print('2. SMOTE diterapkan hanya pada TRAIN              : OK')
print('3. Val & Test di-scaled memakai scaler dari TRAIN : OK')
print()

# CEK NILAI TIDAK VALID
print('=== SANITY CHECKS ===')
print(f'Jumlah NaN di X_train_scaled : {np.isnan(X_train_scaled).sum()}')
print(f'Jumlah NaN di X_train_smote  : {np.isnan(X_train_smote).sum()}')
print(f'Jumlah NaN di X_val_scaled   : {np.isnan(X_val_scaled).sum()}')
print(f'Jumlah NaN di X_test_scaled  : {np.isnan(X_test_scaled).sum()}')
print()

# VALIDASI DISTRIBUSI KELAS
print('=== VALIDASI DISTRIBUSI KELAS ===')
print(f'Train (setelah SMOTE): legit {100 - fraud_rate(y_train_smote):.1f}% vs fraud {fraud_rate(y_train_smote):.1f}%')
print(f'Val   (asli)         : legit {100 - fraud_rate(y_val):.2f}% vs fraud {fraud_rate(y_val):.2f}%')
print(f'Test  (asli)         : legit {100 - fraud_rate(y_test):.2f}% vs fraud {fraud_rate(y_test):.2f}%')

=== CEK DATA LEAKAGE ===
1. StandardScaler di-fit hanya pada TRAIN         : OK
2. SMOTE diterapkan hanya pada TRAIN              : OK
3. Val & Test di-scaled memakai scaler dari TRAIN : OK

=== SANITY CHECKS ===
Jumlah NaN di X_train_scaled : V1            0
V2            0
V3            0
V4            0
V5            0
V6            0
V7            0
V8            0
V9            0
V10           0
V11           0
V12           0
V13           0
V14           0
V15           0
V16           0
V17           0
V18           0
V19           0
V20           0
V21           0
V22           0
V23           0
V24           0
V25           0
V26           0
V27           0
V28           0
Amount_log    0
Time_hour     0
Time_day      0
dtype: int64
Jumlah NaN di X_train_smote  : V1            0
V2            0
V3            0
V4            0
V5            0
V6            0
V7            0
V8            0
V9            0
V10           0
V11           0
V12           0
V13           0
V14     

In [21]:
# Simpan semua artefak ke folder processed
artifacts = {
    'X_train_scaled.pkl': X_train_scaled,
    'X_train_smote.pkl': X_train_smote,
    'X_val_scaled.pkl': X_val_scaled,
    'X_test_scaled.pkl': X_test_scaled,
    'y_train.pkl': y_train,
    'y_train_smote.pkl': y_train_smote,
    'y_val.pkl': y_val,
    'y_test.pkl': y_test,
    'scaler.pkl': scaler,
}

for filename, obj in artifacts.items():
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'Saved: {path}')

# Simpan dataset bersih (deliverable)
clean_csv_path = os.path.join(OUTPUT_DIR, 'cleaned_creditcard.csv')
df_clean.to_csv(clean_csv_path, index=False)
print(f'Saved: {clean_csv_path}')

Saved: /content/drive/MyDrive/Fraud Detection/data/processed/X_train_scaled.pkl
Saved: /content/drive/MyDrive/Fraud Detection/data/processed/X_train_smote.pkl
Saved: /content/drive/MyDrive/Fraud Detection/data/processed/X_val_scaled.pkl
Saved: /content/drive/MyDrive/Fraud Detection/data/processed/X_test_scaled.pkl
Saved: /content/drive/MyDrive/Fraud Detection/data/processed/y_train.pkl
Saved: /content/drive/MyDrive/Fraud Detection/data/processed/y_train_smote.pkl
Saved: /content/drive/MyDrive/Fraud Detection/data/processed/y_val.pkl
Saved: /content/drive/MyDrive/Fraud Detection/data/processed/y_test.pkl
Saved: /content/drive/MyDrive/Fraud Detection/data/processed/scaler.pkl
Saved: /content/drive/MyDrive/Fraud Detection/data/processed/cleaned_creditcard.csv
